# Interactive Bird Dashboard
Explore Newfoundland and Labrador bird observation data, species rarity, seasonal trends, and location hotspots.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('birds.csv')
df['OBSERVATION DATE'] = pd.to_datetime(df['OBSERVATION DATE'])
df['Month'] = df['OBSERVATION DATE'].dt.month
df['OBSERVATION COUNT'] = pd.to_numeric(df['OBSERVATION COUNT'], errors='coerce').fillna(1)

print(f"Loaded {len(df):,} rows, {df['COMMON NAME'].nunique()} unique species")

# using IBA CODE as a proxy for protected area status
# if the row has an IBA code it means the sighting was inside a protected bird area
df['Protected Area Status'] = df['IBA CODE'].apply(
    lambda x: 'Inside Protected Area' if pd.notnull(x) else 'Outside Protected Area'
)

# rarity score: rank species by total observations (percentile), then flip it
# so a species seen rarely ends up close to 100, common ones near 0
species_totals = df.groupby('COMMON NAME')['OBSERVATION COUNT'].sum()
rarity_score_map = 100 - species_totals.rank(pct=True) * 100
df['Rarity Score'] = df['COMMON NAME'].map(rarity_score_map).round(2)

# -0.001 on the left edge so pd.cut doesn't exclude the min value
# had a bug where some "Abundant" birds were getting dropped before i added that
df['Rarity Tier'] = pd.cut(
    df['Rarity Score'],
    bins=[-0.001, 24.999, 49.999, 74.999, 94.999, 100.0],
    labels=['Abundant', 'Common', 'Uncommon', 'Rare', 'Ultra-Rare']
).astype(str)

tier_colors = {
    'Ultra-Rare': '#8E44AD',
    'Rare':       '#E74C3C',
    'Uncommon':   '#E67E22',
    'Common':     '#27AE60',
    'Abundant':   '#3498DB',
}

tier_order = ['Ultra-Rare', 'Rare', 'Uncommon', 'Common', 'Abundant']

# one row per species with its rarity info
species_summary = df.groupby('COMMON NAME').agg(
    Rarity_Score=('Rarity Score', 'first'),
    Rarity_Tier=('Rarity Tier', 'first'),
    Total_Obs=('OBSERVATION COUNT', 'sum'),
).reset_index().sort_values('Rarity_Score', ascending=False)

# global stats for the KPI cards (these update per filter in the callback)
total_obs = int(df['OBSERVATION COUNT'].sum())
unique_species = df['COMMON NAME'].nunique()
iba_pct = round(
    df[df['Protected Area Status'] == 'Inside Protected Area']['OBSERVATION COUNT'].sum()
    / df['OBSERVATION COUNT'].sum() * 100, 1
)
rarest_species = species_summary.iloc[0]['COMMON NAME']

print(f"Total obs: {total_obs:,} | Species: {unique_species} | IBA%: {iba_pct}%")
print(f"Rarest species in dataset: {rarest_species}")

available_birds = sorted(df['COMMON NAME'].unique())
available_counties = sorted(df['COUNTY'].dropna().unique())
available_ibas = ['Inside Protected Area', 'Outside Protected Area']

month_names = {
    1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
    7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'
}

# figure out peak month from raw counts
_peak_month_idx = int(df.groupby('Month')['OBSERVATION COUNT'].sum().idxmax())
PEAK_MONTH_NAME = ['January','February','March','April','May','June',
                   'July','August','September','October','November','December'][_peak_month_idx - 1]

_ur = df[df['Rarity Tier'] == 'Ultra-Rare']
ULTRA_RARE_IBA_PCT = round(
    _ur[_ur['Protected Area Status'] == 'Inside Protected Area']['OBSERVATION COUNT'].sum()
    / max(_ur['OBSERVATION COUNT'].sum(), 1) * 100
)

TOP_COUNTY = (df.groupby('COUNTY')['OBSERVATION COUNT']
              .sum().sort_values(ascending=False).index[0])
OBS_DATE_MIN = df['OBSERVATION DATE'].min().strftime('%b %Y')
OBS_DATE_MAX = df['OBSERVATION DATE'].max().strftime('%b %Y')

# pre-split by month so the map callback doesn't re-filter the whole df every tick
MAP_BY_MONTH = {m: df[df['Month'] == m] for m in range(1, 13)}

print("Data ready")

Loaded 308,370 rows, 300 unique species
Total obs: 20,623,892 | Species: 300 | IBA%: 74.9%
Rarest species in dataset: Western Tanager
Data ready


In [2]:
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State

app = Dash(__name__, suppress_callback_exceptions=True)

# css calc strings for layout heights - arrived at these by trial and error
map_height = 'calc(100vh - 222px)'
sidebar_panel_height = 'calc(50vh - 128px)'
body_height = 'calc(100vh - 140px)'

# reusable style dicts so i don't repeat myself everywhere
card_style = {
    'background': '#2C3E50', 'color': 'white', 'borderRadius': '8px',
    'padding': '8px 14px', 'flex': '1', 'minWidth': '130px', 'textAlign': 'center',
    'boxShadow': '0 2px 6px rgba(0,0,0,.3)', 'display': 'flex',
    'flexDirection': 'column', 'justifyContent': 'center', 'alignItems': 'center'
}
filter_label_style = {
    'color': '#94A3B8', 'fontWeight': 'bold', 'fontSize': '10px',
    'marginBottom': '2px', 'marginTop': '8px',
    'textTransform': 'uppercase', 'letterSpacing': '.5px'
}
chart_card_style = {
    'background': '#2C3E50', 'borderRadius': '8px', 'padding': '10px 12px',
    'boxShadow': '0 2px 8px rgba(0,0,0,.3)',
    'display': 'flex', 'flexDirection': 'column', 'overflow': 'hidden'
}
caption_style = {
    'color': '#94A3B8', 'fontSize': '10px', 'fontStyle': 'italic',
    'padding': '4px 2px 0', 'flexShrink': '0'
}

def make_chart_title(bold_text, light_text):
    return html.Div([
        html.Span(bold_text, style={'color': '#E8F4FD', 'fontWeight': 'bold', 'fontSize': '13px'}),
        html.Span(f'  {light_text}', style={'color': '#64748B', 'fontSize': '11px'}),
    ], style={'flexShrink': '0', 'marginBottom': '6px', 'height': '24px'})


app.layout = html.Div(style={
    'fontFamily': 'Segoe UI, Arial, sans-serif',
    'background': '#1A252F',
    'height': '100vh',
    'overflow': 'hidden',
    'display': 'flex',
    'flexDirection': 'column',
    'boxSizing': 'border-box',
}, children=[

    # top header bar
    html.Div(style={
        'background': 'linear-gradient(135deg, #1ABC9C, #2980B9)',
        'padding': '0 28px', 'height': '50px', 'flexShrink': '0',
        'display': 'flex', 'alignItems': 'center', 'gap': '16px',
    }, children=[
        html.Span('Bird Stories of Newfoundland and Labrador',
                  style={'fontSize': '20px', 'fontWeight': 'bold', 'color': 'white'}),
        html.Span('Muhammed Patel & Luckman Qasim | 202158853 & 202126439',
                  style={'fontSize': '12px', 'color': 'rgba(255,255,255,.72)'}),
    ]),

    # KPI cards row
    html.Div(style={
        'display': 'flex', 'gap': '10px', 'padding': '8px 14px',
        'height': '66px', 'flexShrink': '0', 'boxSizing': 'border-box',
    }, children=[
        html.Div([
            html.Div(id='kpi-total', children=f'{total_obs:,}',
                     style={'fontSize': '20px', 'fontWeight': 'bold', 'color': '#1ABC9C'}),
            html.Div('Total Observations', style={'fontSize': '11px', 'opacity': '.72'}),
        ], style=card_style),
        html.Div([
            html.Div(id='kpi-species', children=f'{unique_species}',
                     style={'fontSize': '20px', 'fontWeight': 'bold', 'color': '#F39C12'}),
            html.Div('Unique Species', style={'fontSize': '11px', 'opacity': '.72'}),
        ], style=card_style),
        html.Div([
            html.Div(id='kpi-iba', children=f'{iba_pct}%',
                     style={'fontSize': '20px', 'fontWeight': 'bold', 'color': '#E74C3C'}),
            html.Div('In Protected Areas', style={'fontSize': '11px', 'opacity': '.72'}),
        ], style=card_style),
        html.Div([
            html.Div(id='kpi-rarest', children=rarest_species,
                     style={
                         'fontSize': '13px', 'fontWeight': 'bold', 'color': '#8E44AD',
                         'overflow': 'hidden', 'textOverflow': 'ellipsis',
                         'whiteSpace': 'nowrap', 'maxWidth': '100%'
                     }),
            html.Div('Rarest Species', style={'fontSize': '11px', 'opacity': '.72'}),
        ], style=card_style),
    ]),

    # scope strip - shows what filters are currently active
    html.Div(id='scope-strip',
             style={
                 'background': '#243342', 'color': '#94A3B8',
                 'padding': '5px 14px', 'fontSize': '11px',
                 'borderBottom': '1px solid #2C3E50', 'flexShrink': '0',
                 'fontStyle': 'italic', 'letterSpacing': '.3px'
             },
             children='Showing: all species - all counties - all tiers - inside + outside Protected Areas'),

    # main body: sidebar + charts
    html.Div(style={
        'display': 'flex', 'height': body_height, 'overflow': 'hidden', 'flexShrink': '0',
    }, children=[

        # filter sidebar
        html.Div(style={
            'width': '200px', 'flexShrink': '0', 'background': '#1E2F3E',
            'padding': '10px 12px', 'overflowY': 'auto',
            'borderRight': '1px solid #2C3E50', 'boxSizing': 'border-box',
            'height': '100%',
        }, children=[
            html.Div('Filters', style={'color': '#1ABC9C', 'fontWeight': 'bold',
                                       'fontSize': '13px', 'marginBottom': '2px'}),

            html.Div('Species', style=filter_label_style),
            dcc.Dropdown(id='species-dropdown',
                         options=[{'label': b, 'value': b} for b in available_birds],
                         value=[], multi=True,
                         placeholder='Type to filter - default: all',
                         style={'fontSize': '11px'}),

            html.Div('County', style=filter_label_style),
            dcc.Dropdown(id='county-dropdown',
                         options=[{'label': c, 'value': c} for c in available_counties],
                         value=[], multi=True,
                         placeholder='Type to filter - default: all',
                         style={'fontSize': '11px'}),

            html.Div('Protected Status', style=filter_label_style),
            dcc.Checklist(id='iba-checklist',
                          options=[{'label': f'  {v}', 'value': v} for v in available_ibas],
                          value=available_ibas,
                          labelStyle={'display': 'block', 'color': '#CBD5E0',
                                      'fontSize': '11px', 'marginBottom': '1px'}),

            html.Div('Rarity Tier', style=filter_label_style),
            dcc.Checklist(id='rarity-tier-filter',
                          options=[{'label': f'  {t}', 'value': t} for t in tier_order],
                          value=tier_order[:],
                          labelStyle={'display': 'block', 'color': '#CBD5E0',
                                      'fontSize': '11px', 'marginBottom': '1px'}),

            html.Div('Protected Area scale', style=filter_label_style),
            html.Div('applies to the bottom-right chart only',
                     style={'color': '#64748B', 'fontSize': '9px', 'marginTop': '0'}),
            dcc.RadioItems(id='scale-radio',
                           options=[{'label': '  Linear', 'value': 'linear'},
                                    {'label': '  Log', 'value': 'log'}],
                           value='linear',
                           labelStyle={'display': 'inline-block', 'color': '#CBD5E0',
                                       'fontSize': '11px', 'marginRight': '8px'}),

            html.Div('Map colour', style=filter_label_style),
            dcc.RadioItems(id='map-color-toggle',
                           options=[{'label': '  Rarity', 'value': 'rarity'},
                                    {'label': '  Species', 'value': 'species'}],
                           value='rarity',
                           labelStyle={'display': 'inline-block', 'color': '#CBD5E0',
                                       'fontSize': '11px', 'marginRight': '8px'}),

            html.Div('Month (map)', style=filter_label_style),
            dcc.Slider(id='month-slider', min=1, max=12, step=1, value=6,
                       marks={m: {'label': month_names[m],
                                  'style': {'color': '#94A3B8', 'fontSize': '9px'}}
                              for m in range(1, 13)},
                       tooltip={'always_visible': False}),
            html.Div(id='month-readout',
                     children=f'Showing: {month_names[6]}',
                     style={'color': '#1ABC9C', 'fontSize': '10px', 'textAlign': 'center',
                            'marginTop': '4px', 'fontWeight': 'bold'}),

            html.Div('Animate', style=filter_label_style),
            html.Button('Play', id='animate-btn', n_clicks=0, style={
                'background': '#1ABC9C', 'color': 'white', 'border': 'none',
                'borderRadius': '5px', 'padding': '5px 0', 'cursor': 'pointer',
                'fontSize': '12px', 'width': '100%', 'marginTop': '3px',
            }),
            dcc.Interval(id='animation-interval', interval=900, disabled=True, n_intervals=0),
        ]),

        # charts area
        html.Div(style={
            'flex': '1', 'minWidth': '0',
            'display': 'flex', 'flexDirection': 'row',
            'padding': '8px', 'gap': '8px',
            'height': '100%', 'boxSizing': 'border-box',
            'overflow': 'hidden',
        }, children=[

            # map (left, takes up most of the width)
            html.Div(style={**chart_card_style, 'flex': '1', 'minWidth': '0', 'height': '100%'}, children=[
                make_chart_title(
                    'Where the birds are right now',
                    'Dot size = how many were seen · colour = how rare the species is'
                ),
                dcc.Graph(id='migration-map',
                          style={'height': map_height},
                          config={'displayModeBar': True, 'scrollZoom': True}),
                html.Div(f'Biggest hotspot overall: {TOP_COUNTY}. Drag the month slider to watch the year unfold.',
                         style=caption_style),
            ]),

            # right column: rarity bar + protected area bar
            html.Div(style={
                'flex': '1', 'minWidth': '0',
                'display': 'flex', 'flexDirection': 'column',
                'gap': '8px', 'height': '100%', 'overflow': 'hidden',
            }, children=[

                html.Div(style={**chart_card_style, 'flex': '1', 'minHeight': '0'}, children=[
                    make_chart_title(
                        'From everyday robins to once-a-year rarities',
                        'Each bar is a species, ranked 0 (everywhere) to 100 (almost never seen)'
                    ),
                    dcc.Graph(id='rarity-distribution',
                              style={'height': sidebar_panel_height},
                              config={'displayModeBar': True}),
                    html.Div('Reading left-to-right: common on the left, once-a-year rarities on the right.',
                             style=caption_style),
                ]),

                html.Div(style={**chart_card_style, 'flex': '1', 'minHeight': '0'}, children=[
                    make_chart_title(
                        'Do protected areas actually shelter rare birds?',
                        f'{int(ULTRA_RARE_IBA_PCT)}% of ultra-rare sightings happen inside a protected area'
                    ),
                    dcc.Graph(id='iba-impact-chart',
                              style={'height': sidebar_panel_height},
                              config={'displayModeBar': True}),
                    html.Div(f'Rare birds cluster in protected areas: {int(ULTRA_RARE_IBA_PCT)}% of ultra-rare sightings are inside one.',
                             style=caption_style),
                ]),
            ]),
        ]),
    ]),
])


# play/pause button controls the interval timer
@app.callback(
    Output('animation-interval', 'disabled'),
    Output('animate-btn', 'children'),
    Output('month-slider', 'value'),
    Input('animate-btn', 'n_clicks'),
    Input('animation-interval', 'n_intervals'),
    State('animation-interval', 'disabled'),
    State('month-slider', 'value'),
    prevent_initial_call=True,
)
def handle_animation(n_clicks, n_intervals, interval_disabled, current_month):
    from dash import ctx
    if ctx.triggered_id == 'animate-btn':
        # toggle between play and pause
        if interval_disabled:
            return False, 'Pause', current_month
        return True, 'Play', current_month
    # auto-advance to next month, wrap around after december
    return False, 'Pause', (current_month % 12) + 1


@app.callback(
    Output('month-readout', 'children'),
    Input('month-slider', 'value'),
)
def update_month_label(m):
    return f'Showing: {month_names[m]}'


@app.callback(
    Output('scope-strip', 'children'),
    Input('species-dropdown', 'value'),
    Input('county-dropdown', 'value'),
    Input('rarity-tier-filter', 'value'),
    Input('iba-checklist', 'value'),
)
def update_scope_strip(sel_species, sel_counties, sel_tiers, sel_iba):
    parts = []
    parts.append(f'{len(sel_species)} species' if sel_species else 'all species')
    parts.append(f'{len(sel_counties)} counties' if sel_counties else 'all counties')
    parts.append(f'{len(sel_tiers)}/5 tiers' if sel_tiers and len(sel_tiers) < 5 else 'all tiers')
    if sel_iba and len(sel_iba) < 2:
        parts.append(sel_iba[0].lower())
    else:
        parts.append('inside + outside Protected Areas')
    return 'Showing: ' + ' - '.join(parts)


# main callback - rebuilds all 3 charts + 4 KPI values whenever any filter changes
@app.callback(
    Output('migration-map', 'figure'),
    Output('rarity-distribution', 'figure'),
    Output('iba-impact-chart', 'figure'),
    Output('kpi-total', 'children'),
    Output('kpi-species', 'children'),
    Output('kpi-iba', 'children'),
    Output('kpi-rarest', 'children'),
    Input('species-dropdown', 'value'),
    Input('county-dropdown', 'value'),
    Input('iba-checklist', 'value'),
    Input('rarity-tier-filter', 'value'),
    Input('scale-radio', 'value'),
    Input('map-color-toggle', 'value'),
    Input('month-slider', 'value'),
)
def update_charts(sel_species, sel_counties, sel_iba, sel_tiers,
                  scale_mode, color_mode, month):

    # color shortcuts
    dark_bg = '#1E2F3E'
    chart_bg = '#243342'
    font_color = '#E8F4FD'
    grid_color = 'rgba(255,255,255,.08)'

    month_label = ['January','February','March','April','May','June',
                   'July','August','September','October','November','December'][month - 1]

    def make_empty_fig(msg):
        f = go.Figure()
        f.update_layout(
            paper_bgcolor=dark_bg, plot_bgcolor=dark_bg, font_color=font_color,
            margin=dict(l=8, r=8, t=8, b=8),
            annotations=[dict(text=msg, showarrow=False,
                              font=dict(size=13, color='#64748B'),
                              xref='paper', yref='paper', x=.5, y=.5)]
        )
        return f

    # apply all filters to main df
    filt = df.copy()
    if sel_species:
        filt = filt[filt['COMMON NAME'].isin(sel_species)]
    if sel_counties:
        filt = filt[filt['COUNTY'].isin(sel_counties)]
    if sel_iba:
        filt = filt[filt['Protected Area Status'].isin(sel_iba)]
    if sel_tiers:
        filt = filt[filt['Rarity Tier'].isin(sel_tiers)]

    # map uses pre-split monthly slice for speed
    filt_map = MAP_BY_MONTH[month].copy()
    if sel_species:
        filt_map = filt_map[filt_map['COMMON NAME'].isin(sel_species)]
    if sel_counties:
        filt_map = filt_map[filt_map['COUNTY'].isin(sel_counties)]
    if sel_iba:
        filt_map = filt_map[filt_map['Protected Area Status'].isin(sel_iba)]
    if sel_tiers:
        filt_map = filt_map[filt_map['Rarity Tier'].isin(sel_tiers)]

    # recalculate KPI values based on filtered data
    if len(filt) > 0:
        kpi_total = f"{int(filt['OBSERVATION COUNT'].sum()):,}"
        kpi_species = f"{filt['COMMON NAME'].nunique()}"
        inside = filt[filt['Protected Area Status'] == 'Inside Protected Area']['OBSERVATION COUNT'].sum()
        total = max(filt['OBSERVATION COUNT'].sum(), 1)
        kpi_iba = f"{inside / total * 100:.1f}%"
        rarity_ranked = (filt.groupby('COMMON NAME')['Rarity Score']
                         .first().sort_values(ascending=False))
        kpi_rarest = rarity_ranked.index[0] if len(rarity_ranked) else '-'
    else:
        kpi_total = '0'
        kpi_species = '0'
        kpi_iba = '-'
        kpi_rarest = '-'

    # MAP 
    if filt_map.empty:
        map_fig = make_empty_fig(
            f'No sightings match your filters for {month_label}. '
            f'Try clearing a filter or moving the month slider.'
        )
    else:
        agg = (filt_map
               .groupby(['COMMON NAME', 'Rarity Tier', 'LATITUDE', 'LONGITUDE'], as_index=False)
               ['OBSERVATION COUNT'].sum())
        
        agg['dot_size'] = np.log1p(agg['OBSERVATION COUNT']) + 1

        color_col = 'Rarity Tier' if color_mode == 'rarity' else 'COMMON NAME'
        color_map = tier_colors if color_mode == 'rarity' else None

        map_fig = px.scatter_map(
            agg, lat='LATITUDE', lon='LONGITUDE',
            color=color_col, color_discrete_map=color_map,
            size='dot_size', size_max=20,
            hover_name='COMMON NAME',
            hover_data={
                'OBSERVATION COUNT': True, 'Rarity Tier': True,
                'dot_size': False, 'LATITUDE': False, 'LONGITUDE': False
            },
            map_style='carto-darkmatter',
            zoom=4.6, center={'lat': 51.5, 'lon': -56.5}, opacity=.82
        )
        map_fig.update_layout(
            paper_bgcolor=dark_bg, font_color=font_color,
            margin=dict(l=0, r=0, t=32, b=0),
            showlegend=(color_mode == 'rarity'),
            legend=dict(bgcolor='rgba(30,47,62,.88)', font_color=font_color,
                        title_font_color=font_color,
                        title_text='Rarity Tier' if color_mode == 'rarity' else 'Species'),
            uirevision='map',
            title=dict(
                text=f'<b>{month_label}</b> - what people are seeing this month',
                font=dict(size=13, color=font_color), x=.01, y=.99,
                xanchor='left', yanchor='top'
            )
        )

    # RARITY BAR CHART 
    if filt.empty:
        rar_fig = make_empty_fig(
            'No species match your filters. Clear a tier or county to see the rarity ladder.'
        )
    else:
        present_species = filt['COMMON NAME'].unique()
        present_df = (species_summary[species_summary['COMMON NAME'].isin(present_species)]
                      .sort_values('Rarity_Score', ascending=True))

        # cap at 40 bars unless the user specifically filtered species
        if sel_species:
            chart_df = present_df
        else:
            chart_df = present_df.tail(40)

        rar_fig = px.bar(
            chart_df, x='Rarity_Score', y='COMMON NAME',
            color='Rarity_Tier', color_discrete_map=tier_colors, orientation='h',
            hover_data={'Rarity_Score': ':.1f', 'Rarity_Tier': True},
            labels={
                'Rarity_Score': 'Rarity Score (0 = common, 100 = rarest)',
                'COMMON NAME': '', 'Rarity_Tier': 'Rarity Tier'
            },
            category_orders={
                'Rarity_Tier': tier_order,
                'COMMON NAME': chart_df['COMMON NAME'].tolist()
            }
        )
        # dotted lines to show where each tier boundary falls
        for threshold, label in [(25, 'Common'), (50, 'Uncommon'), (75, 'Rare'), (95, 'Ultra-Rare')]:
            rar_fig.add_vline(
                x=threshold, line_dash='dot',
                line_color='rgba(255,255,255,.28)',
                annotation_text=label, annotation_position='top',
                annotation_font_color='#94A3B8', annotation_font_size=9
            )
        rar_fig.update_layout(
            paper_bgcolor=dark_bg, plot_bgcolor=chart_bg, font_color=font_color,
            xaxis=dict(range=[0, 100], gridcolor=grid_color, title_font_size=10),
            yaxis=dict(gridcolor=grid_color, tickfont=dict(size=9)),
            margin=dict(l=6, r=14, t=26, b=24),
            legend=dict(bgcolor='rgba(30,47,62,.88)', font_color=font_color,
                        title_font_color=font_color, title_text='Rarity Tier', font_size=10),
            bargap=.12,
            title=dict(
                text=(f'<span style="color:#94A3B8;font-size:10px">'
                      f'Showing {len(chart_df)} of {len(present_species)} species in current selection</span>'),
                x=0.0, y=0.99, xanchor='left'
            )
        )

    # PROTECTED AREA STACKED BAR
    if filt.empty:
        iba_fig = make_empty_fig('No observations to compare. Widen your filters.')
    else:
        piv = (filt.groupby(['Protected Area Status', 'Rarity Tier'])['OBSERVATION COUNT']
               .sum().unstack(fill_value=0))
        present_tiers = [t for t in tier_order if t in piv.columns]
        melted = (piv[present_tiers].reset_index()
                  .melt(id_vars='Protected Area Status',
                        var_name='Rarity Tier', value_name='Observations'))
        iba_fig = px.bar(
            melted, x='Protected Area Status', y='Observations',
            color='Rarity Tier', color_discrete_map=tier_colors, barmode='stack',
            category_orders={'Rarity Tier': tier_order},
            labels={'Observations': 'Total Observations', 'Protected Area Status': ''}
        )
        iba_fig.update_layout(
            paper_bgcolor=dark_bg, plot_bgcolor=chart_bg, font_color=font_color,
            xaxis=dict(tickfont=dict(size=13, color=font_color), gridcolor=grid_color),
            yaxis=dict(type=scale_mode, gridcolor=grid_color,
                       tickformat=',.0f', title_font_size=10),
            margin=dict(l=6, r=6, t=6, b=24),
            legend=dict(bgcolor='rgba(30,47,62,.88)', font_color=font_color,
                        title_font_color=font_color, title_text='Rarity Tier', font_size=10),
            bargap=.30
        )

    return map_fig, rar_fig, iba_fig, kpi_total, kpi_species, kpi_iba, kpi_rarest


if __name__ == '__main__':
    app.run(debug=True, port=8050)
else:
    app.run(debug=False, port=8050, jupyter_mode='inline')


### Basic Code Documentation

#### 2. dashboard.ipynb
This notebook creates a responsive, interactive web application using `Plotly Dash`, allowing users to drill down into the birding dataset.
*   **App Layout**: Designed with a fixed header, a KPI summary row, a collapsible filter sidebar, and three main dynamic chart areas.
*   **Interactive Components (Sidebar)**: Includes Multi-select dropdowns for Species and County, checklists for Protected Status and Rarity Tier, and toggles for chart scaling and map coloring. Importantly, it includes a Month Slider with a Play/Pause animation interval.
*   **Callbacks**:
    1.  `animate`: Controls the interval timer for the month slider, powering the time-lapse animation.
    2.  `_scope`: Dynamically updates the text strip indicating the active filter scope.
    3.  `update_charts`: The core callback. It filters the master dataset (`df` and `MAP_BY_MONTH`) based on all active controls and recalculates the 4 KPI cards. It returns three Plotly graphs:
        *   **`migration-map` (Scatter Mapbox)**: Maps observations for the currently selected month.
        *   **`rarity-distribution` (Horizontal Bar)**: Displays the top species ranked by their 0–100 rarity score.
        *   **`iba-impact-chart` (Stacked Bar)**: Shows observation frequencies grouped by Protected Area status.
